# Knowledge Base Builder

This notebook builds a knowledge base from files in `@HYPERCARE_DB.PUBLIC.DOCS` using:
- **AI_PARSE_DOCUMENT** for PDFs, DOCX, PPTX, XLSX, and PNG files
- **AI_TRANSCRIBE** for MP4 video files
- **Cortex Search Service** for hybrid (vector + keyword) search

### Pipeline
1. Parse text-based documents
2. Parse images
3. Transcribe videos
4. Merge and chunk all content
5. Create Cortex Search Service
6. Run tests

In [ ]:
%%sql -r setup_result
USE ROLE ACCOUNTADMIN;
USE DATABASE HYPERCARE_DB;
USE SCHEMA PUBLIC;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
%%sql -r stage_files
-- List all files in the stage
LIST @HYPERCARE_DB.PUBLIC.DOCS;

## Step 1: Parse Documents (PDF, DOCX, PPTX, XLSX)

In [ ]:
%%sql -r parse_docs_result
CREATE OR REPLACE TABLE HYPERCARE_DB.PUBLIC.RAW_TEXT AS
SELECT
    RELATIVE_PATH,
    SPLIT_PART(RELATIVE_PATH, '.', -1) AS file_type,
    TO_VARCHAR(
        AI_PARSE_DOCUMENT(
            TO_FILE('@HYPERCARE_DB.PUBLIC.DOCS', RELATIVE_PATH),
            {'mode': 'LAYOUT'}
        ):content
    ) AS EXTRACTED_LAYOUT
FROM
    DIRECTORY('@HYPERCARE_DB.PUBLIC.DOCS')
WHERE
    RELATIVE_PATH NOT LIKE '%.mp4'
    AND RELATIVE_PATH NOT LIKE '%.png';

## Step 2: Parse Images (PNG)

In [ ]:
%%sql -r parse_images_result
CREATE OR REPLACE TEMPORARY TABLE HYPERCARE_DB.PUBLIC.RAW_TEXT_IMAGES AS
SELECT
    RELATIVE_PATH,
    'png' AS file_type,
    TO_VARCHAR(
        AI_PARSE_DOCUMENT(
            TO_FILE('@HYPERCARE_DB.PUBLIC.DOCS', RELATIVE_PATH),
            {'mode': 'LAYOUT'}
        ):content
    ) AS EXTRACTED_LAYOUT
FROM
    DIRECTORY('@HYPERCARE_DB.PUBLIC.DOCS')
WHERE
    RELATIVE_PATH LIKE '%.png';

## Step 3: Transcribe Videos (MP4)

In [ ]:
%%sql -r transcribe_result
CREATE OR REPLACE TEMPORARY TABLE HYPERCARE_DB.PUBLIC.RAW_TEXT_VIDEOS AS
SELECT
    RELATIVE_PATH,
    'mp4' AS file_type,
    AI_TRANSCRIBE(
        TO_FILE('@HYPERCARE_DB.PUBLIC.DOCS', RELATIVE_PATH)
    ):text::VARCHAR AS EXTRACTED_LAYOUT
FROM
    DIRECTORY('@HYPERCARE_DB.PUBLIC.DOCS')
WHERE
    RELATIVE_PATH LIKE '%.mp4';

## Step 4: Merge All Sources

In [ ]:
%%sql -r merge_result
-- Merge all parsed content into a single table
CREATE OR REPLACE TABLE HYPERCARE_DB.PUBLIC.RAW_TEXT AS
SELECT RELATIVE_PATH, file_type, EXTRACTED_LAYOUT FROM HYPERCARE_DB.PUBLIC.RAW_TEXT
UNION ALL
SELECT RELATIVE_PATH, file_type, EXTRACTED_LAYOUT FROM HYPERCARE_DB.PUBLIC.RAW_TEXT_IMAGES
UNION ALL
SELECT RELATIVE_PATH, file_type, EXTRACTED_LAYOUT FROM HYPERCARE_DB.PUBLIC.RAW_TEXT_VIDEOS;

## Step 5: Chunk Text

In [ ]:
%%sql -r chunk_result
CREATE OR REPLACE TABLE HYPERCARE_DB.PUBLIC.DOC_CHUNKS AS
SELECT
    relative_path,
    file_type,
    BUILD_SCOPED_FILE_URL('@HYPERCARE_DB.PUBLIC.DOCS', relative_path) AS file_url,
    (
        relative_path || ':\n'
        || COALESCE('Header 1: ' || c.value['headers']['header_1'] || '\n', '')
        || COALESCE('Header 2: ' || c.value['headers']['header_2'] || '\n', '')
        || c.value['chunk']
    ) AS chunk,
    c.value['headers']['header_1']::VARCHAR AS header_1,
    c.value['headers']['header_2']::VARCHAR AS header_2
FROM
    HYPERCARE_DB.PUBLIC.RAW_TEXT,
    LATERAL FLATTEN(SNOWFLAKE.CORTEX.SPLIT_TEXT_MARKDOWN_HEADER(
        EXTRACTED_LAYOUT,
        OBJECT_CONSTRUCT('#', 'header_1', '##', 'header_2'),
        2000,
        300
    )) c
WHERE EXTRACTED_LAYOUT IS NOT NULL AND LENGTH(EXTRACTED_LAYOUT) > 0;

## Step 6: Create Cortex Search Service

In [ ]:
%%sql -r search_service_result
CREATE OR REPLACE CORTEX SEARCH SERVICE HYPERCARE_DB.PUBLIC.KNOWLEDGE_BASE
    ON chunk
    ATTRIBUTES file_type
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 hour'
    AS (
        SELECT
            chunk,
            relative_path,
            file_url,
            file_type,
            header_1,
            header_2
        FROM HYPERCARE_DB.PUBLIC.DOC_CHUNKS
    );

---
## Tests

Validate the knowledge base was built correctly.

In [ ]:
from snowflake.snowpark.context import get_active_session
import json

session = get_active_session()

results = {"passed": 0, "failed": 0, "details": []}

def run_test(name, query, check_fn):
    try:
        df = session.sql(query).collect()
        passed, msg = check_fn(df)
        status = "PASS" if passed else "FAIL"
        results["passed" if passed else "failed"] += 1
        results["details"].append(f"{status}: {name} — {msg}")
    except Exception as e:
        results["failed"] += 1
        results["details"].append(f"FAIL: {name} — Error: {e}")

# Test 1: RAW_TEXT has all expected documents
run_test(
    "All files parsed",
    "SELECT COUNT(*) AS cnt FROM HYPERCARE_DB.PUBLIC.RAW_TEXT",
    lambda df: (df[0]["CNT"] >= 40, f"{df[0]['CNT']} documents parsed (expected >= 40)")
)

# Test 2: All file types are represented
run_test(
    "All file types present",
    "SELECT ARRAY_AGG(DISTINCT file_type) WITHIN GROUP (ORDER BY file_type) AS types FROM HYPERCARE_DB.PUBLIC.RAW_TEXT",
    lambda df: (
        all(t in json.loads(df[0]["TYPES"]) for t in ["pdf", "mp4", "png"]),
        f"File types: {df[0]['TYPES']}"
    )
)

# Test 3: DOC_CHUNKS has reasonable chunk count
run_test(
    "Chunks created",
    "SELECT COUNT(*) AS cnt FROM HYPERCARE_DB.PUBLIC.DOC_CHUNKS",
    lambda df: (df[0]["CNT"] >= 100, f"{df[0]['CNT']} chunks (expected >= 100)")
)

# Test 4: No empty chunks
run_test(
    "No empty chunks",
    "SELECT COUNT(*) AS cnt FROM HYPERCARE_DB.PUBLIC.DOC_CHUNKS WHERE chunk IS NULL OR LENGTH(chunk) = 0",
    lambda df: (df[0]["CNT"] == 0, f"{df[0]['CNT']} empty chunks found")
)

# Test 5: MP4 files were transcribed into chunks
run_test(
    "MP4 transcriptions chunked",
    "SELECT COUNT(*) AS cnt FROM HYPERCARE_DB.PUBLIC.DOC_CHUNKS WHERE file_type = 'mp4'",
    lambda df: (df[0]["CNT"] > 0, f"{df[0]['CNT']} MP4 chunks")
)

# Test 6: PNG files were parsed into chunks
run_test(
    "PNG images chunked",
    "SELECT COUNT(*) AS cnt FROM HYPERCARE_DB.PUBLIC.DOC_CHUNKS WHERE file_type = 'png'",
    lambda df: (df[0]["CNT"] > 0, f"{df[0]['CNT']} PNG chunks")
)

# Test 7: Cortex Search Service exists
run_test(
    "Search service exists",
    "SHOW CORTEX SEARCH SERVICES LIKE 'KNOWLEDGE_BASE' IN SCHEMA HYPERCARE_DB.PUBLIC",
    lambda df: (len(df) == 1, f"Found {len(df)} service(s)")
)

# Test 8: Search returns results
run_test(
    "Search returns results",
    """SELECT PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'HYPERCARE_DB.PUBLIC.KNOWLEDGE_BASE',
            '{"query": "pricing", "columns": ["chunk"], "limit": 3}'
        )
    )['results'] AS results""",
    lambda df: (
        len(json.loads(df[0]["RESULTS"])) > 0,
        f"Search returned {len(json.loads(df[0]['RESULTS']))} results"
    )
)

print(f"\n{'='*50}")
print(f"  TEST RESULTS: {results['passed']} passed, {results['failed']} failed")
print(f"{'='*50}")
for d in results["details"]:
    print(f"  {d}")
print(f"{'='*50}")

assert results["failed"] == 0, f"{results['failed']} test(s) failed"

### Content Relevance Tests

These tests search the knowledge base with real domain questions and verify the expected source documents appear in results.

In [ ]:
import json
from snowflake.snowpark.context import get_active_session

session = get_active_session()

def search_kb(query, limit=5, file_type_filter=None):
    filter_clause = ""
    if file_type_filter:
        filter_clause = f', "filter": {{"@eq": {{"file_type": "{file_type_filter}"}}}}'
    sql = f"""SELECT PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'HYPERCARE_DB.PUBLIC.KNOWLEDGE_BASE',
            '{{
                "query": "{query}",
                "columns": ["chunk", "relative_path", "file_type"],
                "limit": {limit}
                {filter_clause}
            }}'
        )
    )['results'] AS results"""
    row = session.sql(sql).collect()[0]
    return json.loads(row["RESULTS"])

content_results = {"passed": 0, "failed": 0, "details": []}

def content_test(name, query, expected_path_substrings, file_type_filter=None):
    """Search the KB and verify at least one expected source file appears in results."""
    if isinstance(expected_path_substrings, str):
        expected_path_substrings = [expected_path_substrings]
    try:
        hits = search_kb(query, limit=5, file_type_filter=file_type_filter)
        paths = [h["relative_path"] for h in hits]
        found = any(
            exp.lower() in p.lower()
            for p in paths
            for exp in expected_path_substrings
        )
        status = "PASS" if found else "FAIL"
        content_results["passed" if found else "failed"] += 1
        top_file = paths[0] if paths else "no results"
        content_results["details"].append(
            f"{status}: {name}\n"
            f"       Query: '{query}'\n"
            f"       Expected: {expected_path_substrings}  |  Top hit: {top_file}"
        )
    except Exception as e:
        content_results["failed"] += 1
        content_results["details"].append(f"FAIL: {name} — Error: {e}")

# --- PDF content tests ---

content_test(
    "Dynamic pricing strategy setup",
    "How do I set up a dynamic pricing strategy in AQUA Spot?",
    ["How to Create Strategy", "strategy dynamic pricing"]
)

content_test(
    "Empty container SOC rules (PDF)",
    "How are empty containers treated as SOC in AQUA Spot?",
    "AQUA Spot - Empty"
)

content_test(
    "Charge auto vs line flexibility (PDF)",
    "What is the difference between charge auto and line flexibility?",
    "Release V10"
)

content_test(
    "Special equipment on SpotOn (PDF)",
    "How to open special equipments on SpotOn line policy?",
    "Special equipments"
)

content_test(
    "EUP ticket submission for AQUA (DOCX)",
    "How do I raise an EUP support ticket for AQUA Spot?",
    "Hypercare Form"
)

content_test(
    "Release V11 favorites and cloning (PDF)",
    "What new features are in AQUA Spot release V11 like favorites and cloning?",
    ["Release V11", "WEBINAR"]
)

content_test(
    "Onboarding dynamic pricing (PDF)",
    "dynamic pricing onboarding process",
    "ONBOARDING"
)

content_test(
    "Inland rate check (PDF)",
    "How to check an inland rate in AQUA Spot?",
    "inland rate"
)

# --- Video transcript tests (MP4) ---

content_test(
    "Dynamic pricing demo video (MP4)",
    "set up CSA and AquaSpot strategy for rate recommendations",
    "Demo - Dynamic Pricing",
    file_type_filter="mp4"
)

content_test(
    "Mass update video (MP4)",
    "How to use mass update to change charge values on price lines?",
    "Mass Update",
    file_type_filter="mp4"
)

content_test(
    "Mass create video (MP4)",
    "How to create multiple price lines from one screen action?",
    "Mass create",
    file_type_filter="mp4"
)

content_test(
    "Middle East charge setup video (MP4)",
    "How to set up charges for ramp routings using FPOD structure?",
    "Middle East",
    file_type_filter="mp4"
)

# --- PPTX content test ---

content_test(
    "Strategy B market elasticity (PPTX)",
    "What is Strategy B market elasticity and how do rate recommendations work?",
    "strategy dynamic pricing"
)

# --- Print results ---

print(f"\n{'='*60}")
print(f"  CONTENT RELEVANCE TESTS: {content_results['passed']} passed, {content_results['failed']} failed")
print(f"{'='*60}")
for d in content_results["details"]:
    print(f"  {d}")
    print()
print(f"{'='*60}")

assert content_results["failed"] == 0, f"{content_results['failed']} content test(s) failed"